In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import pypsa

INPUT_DIR = Path("inputs") / "tamil_nadu_ra_2025_26"
n_year = pypsa.Network(INPUT_DIR)
component_metadata = pd.read_csv(INPUT_DIR / "component_metadata.csv")
plant_generators = component_metadata.loc[
    component_metadata.component_type == "Generator", "component_id"
]
plant_storage = component_metadata.loc[
    component_metadata.component_type == "StorageUnit", "component_id"
]
assert set(plant_generators) <= set(n_year.generators.index)
assert set(plant_storage) == set(n_year.storage_units.index)
print(f"Loaded {len(plant_generators)} workbook-record generators and {len(plant_storage)} storage units")


# Select the Monday-Sunday week containing the annual demand peak.
annual_demand = n_year.loads_t.p_set.sum(axis=1)
peak_timestamp = annual_demand.idxmax()
week_start = peak_timestamp.normalize() - pd.Timedelta(days=peak_timestamp.weekday())
week_end = week_start + pd.Timedelta(days=7)
week = n_year.snapshots[(n_year.snapshots >= week_start) & (n_year.snapshots < week_end)]

assert len(week) == 168, f"Expected 168 hourly snapshots, found {len(week)}"
n = n_year.copy(snapshots=week)

print(f"Annual peak: {annual_demand.loc[peak_timestamp]:,.2f} MW at {peak_timestamp}")
print(f"Optimizing peak week: {week_start} to {week_end - pd.Timedelta(hours=1)}")

INFO:pypsa.network.io:Imported network 'Tamil Nadu FY 2025-26 single-node UC inputs' has buses, carriers, generators, loads, storage_units


Loaded 276 workbook-record generators and 4 storage units
Annual peak: 19,987.33 MW at 2025-07-11 16:00:00
Optimizing peak week: 2025-07-07 00:00:00 to 2025-07-13 23:00:00


In [2]:
n

PyPSA Network 'Tamil Nadu FY 2025-26 single-node UC inputs'
-----------------------------------------------------------
Components:
 - Bus: 1
 - Carrier: 12
 - Generator: 278
 - Load: 1
 - StorageUnit: 4
Snapshots: 168

Model overview

In [3]:
display(n.generators)

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,min_up_time,min_down_time,up_time_before,down_time_before,ramp_limit_up,ramp_limit_down,ramp_limit_start_up,ramp_limit_shut_down,weight,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
itpcl_or_cuddalore_tpp__unit_01,Tamil_Nadu,PQ,,600.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
itpcl_or_cuddalore_tpp__unit_02,Tamil_Nadu,PQ,,600.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
mettur_tps__unit_01,Tamil_Nadu,PQ,,210.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
mettur_tps__unit_02,Tamil_Nadu,PQ,,210.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
mettur_tps__unit_03,Tamil_Nadu,PQ,,210.00,0.0,False,0.0,inf,NaN,0.55,...,8,8,0,8,0.6,0.6,0.55,0.55,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
thirumurthy_mini_hydel_project__unit_03,Tamil_Nadu,PQ,,0.65,0.0,False,0.0,inf,NaN,0.00,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
vaigai_hydro_power_project__unit_01,Tamil_Nadu,PQ,,3.00,0.0,False,0.0,inf,NaN,0.00,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0
vaigai_hydro_power_project__unit_02,Tamil_Nadu,PQ,,3.00,0.0,False,0.0,inf,NaN,0.00,...,0,0,1,0,NaN,NaN,NaN,NaN,1.0,0.0


In [4]:
display(n.storage_units)

,bus,control,type,p_nom,p_nom_mod,p_nom_extendable,p_nom_min,p_nom_max,p_nom_set,p_min_pu,...,state_of_charge_initial_per_period,state_of_charge_set,cyclic_state_of_charge,cyclic_state_of_charge_per_period,max_hours,efficiency_store,efficiency_dispatch,standing_loss,inflow,p_nom_opt
name,,,,,,,,,,,,,,,,,,,,,
kadmparai_power_house__unit_01,Tamil_Nadu,PQ,,100.0,0.0,False,0.0,inf,NaN,-0.95,...,False,NaN,True,False,6.0,0.894427,0.894427,0.0,0.0,0.0
kadmparai_power_house__unit_02,Tamil_Nadu,PQ,,100.0,0.0,False,0.0,inf,NaN,-0.95,...,False,NaN,True,False,6.0,0.894427,0.894427,0.0,0.0,0.0
kadmparai_power_house__unit_03,Tamil_Nadu,PQ,,100.0,0.0,False,0.0,inf,NaN,-0.95,...,False,NaN,True,False,6.0,0.894427,0.894427,0.0,0.0,0.0
kadmparai_power_house__unit_04,Tamil_Nadu,PQ,,100.0,0.0,False,0.0,inf,NaN,-0.95,...,False,NaN,True,False,6.0,0.894427,0.894427,0.0,0.0,0.0


Random forced-outage scenario

In [5]:
def make_forced_outage_scenario(
    base_network,
    snapshots,
    carriers=("coal", "nuclear", "oil_gas"),
    n_initial_outages=2,
    n_random_faults=8,
    peak_outage_time=None,
    n_peak_faults=3,
    repair_time_hours=(4, 24),
    seed=20250812,
):
    """Return a network with reproducible unit-level forced outages."""
    n = base_network.copy(snapshots=snapshots)
    rng = np.random.default_rng(seed)

    eligible = n.generators.index[
        n.generators.carrier.isin(carriers) & n.generators.active
    ]
    n_events = n_initial_outages + n_random_faults
    if n_events > len(eligible):
        raise ValueError(
            f"Requested {n_events} outages, but only {len(eligible)} eligible units exist."
        )
    if n_random_faults and len(n.snapshots) < 2:
        raise ValueError("Random faults require at least two snapshots.")
    if not 0 <= n_peak_faults <= n_random_faults:
        raise ValueError("n_peak_faults must be between zero and n_random_faults.")
    if n_peak_faults:
        peak_outage_time = pd.Timestamp(peak_outage_time)
        if peak_outage_time not in n.snapshots:
            raise ValueError("peak_outage_time must be one of the selected snapshots.")

    selected = rng.choice(eligible.to_numpy(), size=n_events, replace=False)
    availability = pd.DataFrame(1.0, index=n.snapshots, columns=eligible)
    records = []

    for position, unit in enumerate(selected):
        duration_h = int(
            rng.integers(repair_time_hours[0], repair_time_hours[1] + 1)
        )
        random_fault_position = position - n_initial_outages

        if position < n_initial_outages:
            fault_start = n.snapshots[0]
            outage_type = "initial"
        elif random_fault_position < n_peak_faults:
            # Randomize the start while guaranteeing that repair occurs after
            # the annual-demand peak snapshot.
            duration = pd.Timedelta(hours=duration_h)
            candidates = n.snapshots[
                (n.snapshots <= peak_outage_time)
                & (n.snapshots + duration > peak_outage_time)
            ]
            candidates = candidates[candidates > n.snapshots[0]]
            if candidates.empty:
                raise ValueError("No valid start time can overlap peak_outage_time.")
            fault_start = candidates[int(rng.integers(0, len(candidates)))]
            outage_type = "peak-overlap"
        else:
            start_position = int(rng.integers(1, len(n.snapshots)))
            fault_start = n.snapshots[start_position]
            outage_type = "random"

        available_again = fault_start + pd.Timedelta(hours=duration_h)
        outage_period = (n.snapshots >= fault_start) & (n.snapshots < available_again)
        availability.loc[outage_period, unit] = 0.0

        records.append(
            {
                "generator": unit,
                "carrier": n.generators.at[unit, "carrier"],
                "capacity_MW": n.generators.at[unit, "p_nom"],
                "outage_type": outage_type,
                "fault_start": fault_start,
                "available_again": available_again,
                "duration_h": duration_h,
            }
        )

    # Multiply the original dispatch bounds by forced availability. This preserves
    # existing availability profiles and also keeps non-committable units feasible.
    for unit in eligible:
        for attribute in ("p_max_pu", "p_min_pu"):
            time_series = getattr(n.generators_t, attribute)
            static_value = float(n.generators.at[unit, attribute])
            if unit in time_series.columns:
                baseline = time_series[unit].reindex(n.snapshots).fillna(static_value)
            else:
                baseline = pd.Series(static_value, index=n.snapshots)
            time_series[unit] = baseline * availability[unit]

    if n_peak_faults:
        peak_units = selected[
            n_initial_outages:n_initial_outages + n_peak_faults
        ]
        assert (availability.loc[peak_outage_time, peak_units] == 0.0).all()

    outage_log = (
        pd.DataFrame(records).sort_values("fault_start").reset_index(drop=True)
    )
    return n, outage_log, availability


n, outage_log, forced_availability = make_forced_outage_scenario(
    n_year,
    week,
    carriers=("coal", "nuclear", "oil_gas"),
    n_initial_outages=2,
    n_random_faults=8,
    peak_outage_time=peak_timestamp,
    n_peak_faults=3,
    repair_time_hours=(4, 24),
    seed=20250812,
)

display(outage_log)

,generator,carrier,capacity_MW,outage_type,fault_start,available_again,duration_h
0,neyveli_tps_ii__unit_05,coal,210.0,initial,2025-07-07 00:00:00,2025-07-07 05:00:00,5
1,tuticorin_joint_venture__unit_02,coal,500.0,initial,2025-07-07 00:00:00,2025-07-07 15:00:00,15
2,tuticorin_tps__unit_04,coal,210.0,random,2025-07-07 04:00:00,2025-07-07 20:00:00,16
3,neyveli_ext_tps_ii__unit_01,coal,250.0,random,2025-07-08 09:00:00,2025-07-08 21:00:00,12
4,basin_bridge_gas_turbine_power_station__unit_02,oil_gas,30.0,random,2025-07-08 09:00:00,2025-07-09 02:00:00,17
5,tuticorin_tps__unit_02,coal,210.0,random,2025-07-08 11:00:00,2025-07-08 16:00:00,5
6,vallur_tpp__unit_02,coal,500.0,random,2025-07-10 16:00:00,2025-07-10 20:00:00,4
7,mettur_tps__unit_05,coal,600.0,peak-overlap,2025-07-11 12:00:00,2025-07-11 22:00:00,10
8,valuthur_gas_turbine_power_station__unit_01,oil_gas,60.0,peak-overlap,2025-07-11 12:00:00,2025-07-11 18:00:00,6
9,mettur_tps__unit_03,coal,210.0,peak-overlap,2025-07-11 13:00:00,2025-07-12 00:00:00,11


Running

In [6]:
status, termination_condition = n.optimize(
    solver_name="highs",
)

if termination_condition != "optimal":
    raise RuntimeError(f"Optimization failed: {status}, {termination_condition}")

print(f"Optimization: {status} ({termination_condition})")

C:\Users\b076218\AppData\Local\Temp\ipykernel_22424\4193807742.py:1: FutureWarning: The default value of `include_objective_constant` will change from True to False in version 2.0. Set `include_objective_constant` explicitly to suppress this warning. Using False improves LP numerical conditioning by not including the objective constant as a variable.
  status, termination_condition = n.optimize(
       'mettur_tps__unit_01', 'mettur_tps__unit_02', 'mettur_tps__unit_03',
       'mettur_tps__unit_04', 'mettur_tps__unit_05', 'muthiara_tpp__unit_01',
       'muthiara_tpp__unit_02', 'neyveli_ext_tps_i__unit_01',
       ...
       'shakti_sugars_ltd_tamil_nadu_erode_32__unit_01',
       'shriraam_city_union_finance_ltd_tamil_nadu_thanjavur_7_5__unit_01',
       'shriram_investments_ltd_tamil_nadu_dindigul_7_5__unit_01',
       'sripathi_paper_board_p_ltd_tamil_nadu_virudhunagar_4_95_09_03_20__unit_01',
       'subashri_bio_energy_p_ltd_tamil_nadu_namakkal_2_5__unit_01',
       'subramania_si

Optimization: ok (optimal)


In [7]:
unserved_mwh = (
    n.generators_t.p["unserved_energy"] * n.snapshot_weightings.generators
).sum()

forced_outage_mw = (
    (1.0 - forced_availability)
    .mul(n.generators.loc[forced_availability.columns, "p_nom"], axis=1)
    .sum(axis=1)
)

print(f"Unserved energy: {unserved_mwh:,.2f} MWh")
print(f"Maximum simultaneous affected nameplate capacity: "
      f"{forced_outage_mw.max():,.2f} MW")

Unserved energy: 0.00 MWh
Maximum simultaneous affected nameplate capacity: 920.00 MW


In [8]:
n.export_to_netcdf("tamil_nadu_2025_26.nc")


INFO:pypsa.network.io:Exported network 'Tamil Nadu FY 2025-26 single-node UC inputs' saved to 'tamil_nadu_2025_26.nc contains: carriers, loads, buses, storage_units, generators, sub_networks


<xarray.Dataset> Size: 2MB
Dimensions:                               (snapshots: 168, carriers_i: 12,
                                           loads_i: 1, loads_t_p_set_i: 1,
                                           loads_t_p_i: 1, buses_i: 1,
                                           buses_t_p_i: 1, storage_units_i: 4,
                                           storage_units_t_p_i: 4,
                                           storage_units_t_p_dispatch_i: 4,
                                           ...
                                           generators_t_p_max_pu_i: 219,
                                           generators_t_p_i: 277,
                                           generators_t_status_i: 244,
                                           generators_t_start_up_i: 278,
                                           generators_t_shut_down_i: 278,
                                           sub_networks_i: 1)
Coordinates: (12/20)
  * snapshots                             (snapshots) int64 1kB 0 1 ... 166 167
  * carriers_i                            (carriers_i) object 96B 'market_imp...
  * loads_i                               (loads_i) object 8B 'Tamil_Nadu_dem...
  * loads_t_p_set_i                       (loads_t_p_set_i) object 8B 'Tamil_...
  * loads_t_p_i                           (loads_t_p_i) object 8B 'Tamil_Nadu...
  * buses_i                               (buses_i) object 8B 'Tamil_Nadu'
    ...                                    ...
  * generators_t_p_max_pu_i               (generators_t_p_max_pu_i) object 2kB ...
  * generators_t_p_i                      (generators_t_p_i) object 2kB 'itpc...
  * generators_t_status_i                 (generators_t_status_i) object 2kB ...
  * generators_t_start_up_i               (generators_t_start_up_i) object 2kB ...
  * generators_t_shut_down_i              (generators_t_shut_down_i) object 2kB ...
  * sub_networks_i                        (sub_networks_i) object 8B '0'
Data variables: (12/60)
    snapshots_snapshot                    (snapshots) datetime64[us] 1kB 2025...
    snapshots_objective                   (snapshots) float64 1kB 1.0 ... 1.0
    snapshots_stores                      (snapshots) float64 1kB 1.0 ... 1.0
    snapshots_generators                  (snapshots) float64 1kB 1.0 ... 1.0
    carriers_nice_name                    (carriers_i) object 96B 'market imp...
    loads_bus                             (loads_i) object 8B 'Tamil_Nadu'
    ...                                    ...
    generators_t_p                        (snapshots, generators_t_p_i) float64 372kB ...
    generators_t_status                   (snapshots, generators_t_status_i) float64 328kB ...
    generators_t_start_up                 (snapshots, generators_t_start_up_i) float64 374kB ...
    generators_t_shut_down                (snapshots, generators_t_shut_down_i) float64 374kB ...
    sub_networks_slack_bus                (sub_networks_i) object 8B 'Tamil_N...
    sub_networks_obj                      (sub_networks_i) float64 8B nan
Attributes:
    network__linearized_uc:       0
    network__multi_invest:        0
    network__objective:           6772601147.517689
    network__objective_constant:  0.0
    network_name:                 Tamil Nadu FY 2025-26 single-node UC inputs
    network_pypsa_version:        1.2.4
    network_srid:                 4326
    crs:                          {"_crs": "GEOGCRS[\"WGS 84\",ENSEMBLE[\"Wor...
    meta:                         {}